The electric field in a capacitor inspired by Joachim Schöberl

In [ ]:
from netgen.meshing import Mesh
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

In [ ]:
def CapacitorGeometry(box_size, el_pos_wid, el_pos_h, el_neg_wid, 
                      el_neg_h, diel_wid, diel_h):

    air = MoveTo(0, 0).RectangleC(box_size, box_size).Face()
    air.edges.name = "Outer"
    air.faces.name = "air"

    electrode_positive = MoveTo(0, 1).RectangleC(el_pos_wid, el_pos_h).Face()
    electrode_positive.edges.name = "electrode_positive"
    electrode_positive.faces.name = "electrode_positive"

    electrode_negative = MoveTo(0, -1).RectangleC(el_neg_wid, el_neg_h).Face()
    electrode_negative.edges.name = "electrode_negative"
    electrode_negative.faces.name = "electrode_negative"

    dielectric = MoveTo(0, 0).RectangleC(diel_wid, diel_h).Face()
    dielectric.faces.name = "dielectric"

    shape = Glue([air - dielectric, dielectric])
    shape = shape - electrode_positive - electrode_negative
    
    return shape


def CapacitorMesh(shape, h_max):
    
    mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=h_max))

    return mesh


def CapacitorSolver(mesh, FE_order, epsr):

    fes_phi = H1(mesh, order=FE_order, dirichlet="el.*")
    fes_E = HCurl(mesh, order=FE_order-1)

    u = fes_phi.TrialFunction()
    v = fes_phi.TestFunction()

    potential_gf = GridFunction(fes_phi)
    potential_gf.Interpolate(mesh.BoundaryCF({"electrode_positive":1, "electrode_negative":-1 }), mesh.Boundaries(".*"))

    a = BilinearForm(epsr*grad(u)*grad(v)*dx).Assemble()
    
    inv = a.mat.Inverse(freedofs=fes_phi.FreeDofs())
    potential_gf.vec.data -= inv@a.mat * potential_gf.vec

    return potential_gf, fes_phi, fes_E

In [ ]:
box_size = 30
el_pos_wid, el_pos_h = 5, 0.5
el_neg_wid, el_neg_h = 5, 0.5
diel_wid, diel_h = 4, 1.5

h_max = 0.5
FE_order = 2

geo = CapacitorGeometry(box_size, el_pos_wid, el_pos_h, el_neg_wid, el_neg_h, diel_wid, diel_h)
Draw(geo);

In [ ]:
mesh = CapacitorMesh(geo, h_max)

Draw (mesh);

In [ ]:
epsr_air, epsr_dielectric = 1.0, 4.0
epsr = mesh.MaterialCF({"air": epsr_air, "dielectric": epsr_dielectric})

Draw(epsr, mesh);

In [ ]:
potential_gf, fes_phi, fes_E = CapacitorSolver(mesh, FE_order, epsr)

In [ ]:
Draw (potential_gf, deformation=True, scale=5);

In [ ]:
E_gf = GridFunction(fes_E)
E_gf.Set(-grad(potential_gf))

In [ ]:
Draw (E_gf, mesh, vectors= {"grid_size": 100});

In [ ]:
Draw (Norm(E_gf), mesh, deformation=True, min=0, max=2);

In [ ]:
N = 20
margin = 2

x_min, x_max = -el_pos_wid/2 - margin, el_pos_wid/2 + margin
y_min, y_max = 0, el_pos_h/2 + margin
z = 0


p = [(
     x_min + (x_max - x_min)*i/N,
     y_min + (y_max - y_min)*j/N,
     z
    )
    for i in range(N)
    for j in range(N)
]

fieldlines = E_gf._BuildFieldLines(mesh, p, num_fieldlines=400, length=4)

Draw(E_gf, mesh, "Electric field E", 
     draw_vol=True, 
     draw_surf=True, 
     objects=[fieldlines],
     autoscale=True, 
     min = 0, 
     max = 1, 
     settings={"Objects": {"Surface": False}}
     );